# Preparação da série diária de Portugal continental

Este notebook constrói a base partilhada pela análise exploratória e pela
futura modelação. Lê os CSV de origem na raiz do projeto, seleciona os 18
distritos de Portugal continental, reconcilia as contagens por tipo e
agrega a meteorologia diária com os pesos históricos documentados.

O resultado é gravado em `analise/dados/serie_diaria_continente.csv`.
A coluna `periodo` identifica desenvolvimento, validação, teste reservado
e os anos em que falta a fonte de acidentes. A base conserva a lacuna no
calendário. A meteorologia do dia-alvo é reanálise ERA5.

In [ ]:
from pathlib import Path
import sys
import unicodedata

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "aquisicao").is_dir()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Executar o notebook dentro da pasta do projeto.")

sys.path.insert(0, str(ROOT))
from aquisicao.obter_meteo_diaria import DESTINO as METEO, DISTRITOS

ACIDENTES = ROOT / "data_agregada" / "acidentes.csv"
if not METEO.is_file():
    raise FileNotFoundError("Executar python aquisicao/obter_meteo_diaria.py antes da EDA.")

ANOS_TREINO = list(range(2010, 2020))
ANOS_FONTE = ANOS_TREINO + [2023, 2024]
TIPOS = ["Despiste", "Colisão", "Atropelamento"]


def chave_area(valor):
    normalizado = unicodedata.normalize("NFKD", valor.strip().casefold())
    return "".join(c for c in normalizado if not unicodedata.combining(c))


areas_continentais = {chave_area(nome) for nome in DISTRITOS}

## 1. Construção e auditoria das contagens

O filtro usa os 18 distritos continentais, com normalização de maiúsculas e acentos, antes da agregação diária. Nos ficheiros de origem, Açores e Madeira surgem em 2023–2024. A exclusão dá o mesmo recorte geográfico a todos os períodos.

As modalidades detalhadas observadas são mapeadas para as três famílias usando a classificação de `Natureza` em `docs/dicionarios/acidentes.md`. Atropelamentos de animais e com fuga entram na família de atropelamentos. A unidade a contar é a chave `(ano, Id. Acidente)`. Nos anos com ficheiro, um dia sem registos recebe zero. O calendário completo preserva 2020–2022 como ausentes e impede que os atrasos façam a ponte de 2019 para 2023.

In [ ]:
acidentes = pd.read_csv(
    ACIDENTES,
    usecols=["ano", "Id. Acidente", "Datahora", "Distrito", "Natureza"],
    dtype={"Id. Acidente": "string"},
)
acidentes["area_id"] = acidentes["Distrito"].map(chave_area, na_action="ignore")
no_continente = acidentes["area_id"].isin(areas_continentais)
fora_continente = acidentes.loc[~no_continente]
assert fora_continente["area_id"].str.startswith("ilha ").fillna(False).all(), (
    "Rever os rótulos de distrito antes de excluir registos."
)
display(fora_continente.groupby("ano").size().rename("Registos de Açores e Madeira excluídos").to_frame())
acidentes = acidentes.loc[no_continente].copy()
assert set(acidentes["area_id"]) == areas_continentais
assert set(acidentes["ano"]) == set(ANOS_FONTE), "Rever cobertura de anos da fonte."
assert acidentes["Id. Acidente"].notna().all() and acidentes["Id. Acidente"].str.strip().ne("").all()
assert not acidentes.duplicated(["ano", "Id. Acidente"]).any(), "Investigar chaves repetidas."

datahora = pd.to_datetime(
    acidentes["Datahora"].str.strip(), format="%Y:%m:%d %H:%M:%S", errors="coerce"
)
assert datahora.notna().all() and datahora.dt.year.eq(acidentes["ano"]).all()
acidentes["data"] = datahora.dt.normalize()

modalidades = {
    "Despiste": [
        "Despiste simples", "Despiste com capotamento", "Despiste sem dispositivo de retenção",
        "Despiste com colisão com veículo imobil. ou obstáculo",
        "Despiste com dispositivo de retenção", "Despiste com fuga",
        "Despiste com transposição do dispositivo de retenção lateral",
    ],
    "Colisão": [
        "Colisão lateral com outro veículo em movimento",
        "Colisão traseira com outro veículo em movimento", "Colisão frontal",
        "Colisão com outras situações", "Colisão com veiculo ou obstáculo na faixa de rodagem",
        "Colisão choque em cadeia", "Colisão com fuga",
    ],
    "Atropelamento": [
        "Atropelamento de peões", "Atropelamento com fuga", "Atropelamento de animais",
    ],
}
mapa = {modalidade: tipo for tipo, nomes in modalidades.items() for modalidade in nomes}
acidentes["natureza"] = acidentes["Natureza"].str.strip()
acidentes["tipo"] = acidentes["natureza"].map(mapa)
assert acidentes["tipo"].notna().all(), acidentes.loc[acidentes["tipo"].isna(), "natureza"].unique()

calendario = pd.date_range("2010-01-01", "2024-12-31", freq="D", name="data")
observado = calendario.year.isin(ANOS_FONTE)
contagens = pd.crosstab(acidentes["data"], acidentes["tipo"])
contagens = contagens.reindex(index=calendario, columns=TIPOS)
contagens.loc[observado] = contagens.loc[observado].fillna(0)
contagens = contagens.astype("Int64")
assert contagens.index.is_unique and contagens.notna().all(axis=1).eq(observado).all()
assert contagens.ge(0).fillna(True).to_numpy().all()
assert contagens.sum().sum() == len(acidentes)
assert contagens.sum().reindex(TIPOS).astype(int).tolist() == (
    acidentes["tipo"].value_counts().reindex(TIPOS).tolist()
)

acidentes_treino = acidentes[acidentes["ano"].isin(ANOS_TREINO)]
display(acidentes_treino.groupby(["tipo", "natureza"]).size().rename("Acidentes no treino").to_frame())
display(pd.DataFrame({
    "Período": ["2010–2019", "2020–2022", "2023", "2024"],
    "Estado": ["Desenvolvimento", "Sem ficheiro de acidentes", "Validação temporal", "Teste reservado"],
    "Dias": [calendario.year.isin(ANOS_TREINO).sum(), (~observado).sum(), (calendario.year == 2023).sum(),
             (calendario.year == 2024).sum()],
}))
print(f"{len(acidentes):,} acidentes reconciliados em {observado.sum():,} dias observados.")

## 2. Meteorologia e luz em Portugal continental

A fonte ERA5 fornece um ponto por distrito do continente, referenciado pelo IPMA. A média para o continente usa pesos fixos com base na proporção de acidentes registados por área em **2010–2014**. O objetivo é dar maior representação ao clima das áreas onde historicamente se concentram os acidentes. Estes pesos usam um período anterior a todas as validações e ficam fixos em todo o estudo. Representam a distribuição histórica dos registos, sem medir o volume de tráfego ou a exposição em cada área.

A precipitação para o continente é uma média ponderada dos totais diários locais, em mm. A extensão da chuva é a percentagem do peso histórico em áreas com pelo menos 1 mm nesse dia. Esse limiar exploratório distingue precipitação ligeira de um dia chuvoso no ponto, e a percentagem é ponderada pelos registos históricos. O vento é a média ponderada dos máximos locais. Mostramos também a diferença média face a pesos iguais entre os 18 distritos.

In [ ]:
meteo = pd.read_csv(METEO, parse_dates=["data"])
meteo["area_id"] = meteo["area"].map(chave_area)
assert meteo["data"].notna().all() and not meteo.duplicated(["area_id", "data"]).any()
assert set(meteo["data"]) == set(calendario)
assert set(meteo["area_id"]) == areas_continentais
assert meteo["tipo_area"].eq("distrito").all()
assert meteo.groupby("area_id")["data"].size().eq(len(calendario)).all()

meteo["duracao_dia_h"] = meteo["duracao_dia_s"] / 3600
meteo["sol_direto_h"] = meteo["horas_sol_s"] / 3600
limites = {
    "temperatura_media_c": (-50, 60), "precipitacao_total_mm": (0, np.inf),
    "vento_max_kmh": (0, np.inf), "nebulosidade_media_pct": (0, 100),
    "duracao_dia_h": (0, 24), "sol_direto_h": (0, 24),
    "radiacao_total_mj_m2": (0, np.inf),
}
qualidade = pd.DataFrame({
    "Omissos": meteo[list(limites)].isna().sum(),
    "Fora de limites": {
        c: (~meteo[c].between(*intervalo) & meteo[c].notna()).sum()
        for c, intervalo in limites.items()
    },
})
display(qualidade)
assert qualidade.to_numpy().sum() == 0, "Investigar cobertura e valores antes de agregar."

pesos = acidentes.loc[acidentes["ano"].between(2010, 2014), "area_id"].value_counts(normalize=True)
assert set(pesos.index) == areas_continentais and np.isclose(pesos.sum(), 1)
meteo["peso"] = meteo["area_id"].map(pesos)
assert np.allclose(meteo.groupby("data")["peso"].sum(), 1)

meteo["areas_com_chuva_pct"] = 100.0 * meteo["precipitacao_total_mm"].ge(1.0)
meteorologicas = [*limites, "areas_com_chuva_pct"]
meteo_continente = meteo[meteorologicas].mul(meteo["peso"], axis=0).groupby(meteo["data"]).sum()
meteo_pesos_iguais = meteo.groupby("data")[meteorologicas].mean()
diferenca = (meteo_continente - meteo_pesos_iguais).abs().loc["2010":"2019"].mean()
display(diferenca.round(3).rename("Diferença absoluta média vs pesos iguais (treino)").to_frame())
display(
    meteo[["area", "peso"]].drop_duplicates().sort_values("peso", ascending=False)
    .assign(peso_pct=lambda d: (100 * d["peso"]).round(2)).drop(columns="peso").reset_index(drop=True)
)
print(f"{len(meteo):,} registos de meteorologia diária em {meteo['area_id'].nunique()} áreas.")

## 3. Exportação para a análise e modelação

A tabela tem uma linha por dia civil, três contagens e os oito indicadores
meteorológicos e solares. O teste mantém-se reservado. Os atrasos e as médias
móveis são calculados a partir da data em cada análise, para que o horizonte
e a disponibilidade dos registos fiquem explícitos.

In [ ]:
base = contagens.join(meteo_continente, validate="one_to_one")
base["periodo"] = "sem_dados"
base.loc["2010":"2019", "periodo"] = "desenvolvimento"
base.loc["2023", "periodo"] = "validacao"
base.loc["2024", "periodo"] = "teste_reservado"
base = base[["periodo", *TIPOS, *meteorologicas]]

assert base.index.equals(calendario)
assert base[TIPOS].notna().all(axis=1).eq(base["periodo"].ne("sem_dados")).all()
assert base[meteorologicas].notna().all().all()

destino = ROOT / "analise" / "dados" / "serie_diaria_continente.csv"
destino.parent.mkdir(parents=True, exist_ok=True)
temporario = destino.with_suffix(".tmp")
base.to_csv(temporario, encoding="utf-8", date_format="%Y-%m-%d")
temporario.replace(destino)

display(base["periodo"].value_counts(sort=False).rename("Dias").to_frame())
print(f"Base diária gravada em {destino.relative_to(ROOT)}")